# SQL Magic — Querying `flights.db` Directly in Jupyter

**Author:** Daniel Gideon Kimani  
**Dataset:** US Domestic Flights · Jan 2008 · 605,765 rows  
**Library:** `ipython-sql` (`%sql` / `%%sql` magic)

---

## What is SQL Magic?

`ipython-sql` lets you write **raw SQL directly in notebook cells** using magic commands:

| Syntax | Use |
|--------|-----|
| `%load_ext sql` | Load the extension once |
| `%sql SELECT ...` | Single-line SQL query |
| `%%sql` | Multi-line SQL cell |
| `%sql engine` | Connect via SQLAlchemy URL |

Results render as **interactive tables** and can be saved back to Python variables.

---


## Step 1 — Install & Load the Extension

In [4]:
# Install if needed (run once)
# !pip install ipython-sql sqlalchemy --quiet

%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## Step 2 — Connect to the Database

In [5]:
# SQLAlchemy connection string format:
#   dialect+driver://path_to_db
# For SQLite: sqlite:////absolute/path/to/file.db

%sql sqlite:///flights.db

print("Connected to flights.db")

Connected to flights.db


## Step 3 — Single-line Queries with `%sql`

Use `%sql` for quick one-liners. Results display as a formatted table automatically.


In [6]:
# Count total flights
%sql SELECT COUNT(*) AS total_flights FROM flights;

 * sqlite:///flights.db
Done.


total_flights
605765


In [7]:
# Date range of the dataset
%sql SELECT MIN(Date) AS first_date, MAX(Date) AS last_date FROM flights;

 * sqlite:///flights.db


Done.


first_date,last_date
2008/1/1,2008/1/9


In [8]:
# All tables in the database
%sql SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:///flights.db
Done.


name
airports
carriers
flights
planes
sysdiagrams


## Step 4 — Multi-line Queries with `%%sql`

`%%sql` turns the entire cell into a SQL block. Ideal for readable, complex queries.


In [20]:
%%sql 


-- Flight volume and average delay by day of week
SELECT
    CASE DayOfWeek
        WHEN 1 THEN 'Monday'
        WHEN 2 THEN 'Tuesday'
        WHEN 3 THEN 'Wednesday'
        WHEN 4 THEN 'Thursday'
        WHEN 5 THEN 'Friday'
        WHEN 6 THEN 'Saturday'
        WHEN 7 THEN 'Sunday'
    END                      AS Day,
    COUNT(*)                 AS Flights,
    ROUND(AVG(ArrDelay), 2) AS AvgArrDelay,
    ROUND(AVG(DepDelay), 2) AS AvgDepDelay
FROM flights
WHERE ArrDelay IS NOT NULL
GROUP BY DayOfWeek
ORDER BY DayOfWeek;

 * sqlite:///flights.db
Done.


Day,Flights,AvgArrDelay,AvgDepDelay
Monday,77999,10.59,11.82
Tuesday,94093,11.38,11.84
Wednesday,97279,7.78,9.47
Thursday,98736,13.98,13.85
Friday,79742,10.21,11.48
Saturday,64998,6.12,8.83
Sunday,74283,9.91,12.13


In [21]:
%%sql

-- Top 10 carriers by flight volume with on-time rate
SELECT
    f.UniqueCarrier                           AS Carrier,
    c.Description                             AS Name,
    COUNT(*)                                  AS TotalFlights,
    ROUND(AVG(f.ArrDelay), 2)                AS AvgDelay,
    ROUND(
        100.0 * SUM(CASE WHEN f.ArrDelay <= 0 THEN 1 ELSE 0 END)
        / COUNT(*), 2
    )                                         AS OnTimePct
FROM flights f
LEFT JOIN carriers c ON f.UniqueCarrier = c.Code
WHERE f.ArrDelay IS NOT NULL
GROUP BY f.UniqueCarrier
HAVING TotalFlights > 5000
ORDER BY OnTimePct DESC
LIMIT 10;

 * sqlite:///flights.db
Done.


Carrier,Name,TotalFlights,AvgDelay,OnTimePct
US,US Airways Inc. (Merged with America West 9/05. Reporting for both starting 10/07.),38513,1.82,61.5
B6,JetBlue Airways,16207,4.86,59.26
DL,Delta Air Lines Inc.,36971,4.44,58.05
WN,Southwest Airlines Co.,100107,6.87,55.9
YV,Mesa Airlines Inc.,20666,14.5,54.02
AS,Alaska Airlines Inc.,12289,8.18,53.93
EV,Atlantic Southeast Airlines,21945,9.17,53.21
XE,Expressjet Airlines Inc.,34368,9.89,52.42
FL,AirTran Airways Corporation,20234,7.22,52.15
9E,Pinnacle Airlines Inc.,21624,12.27,52.14


## Step 5 — Save Results to a Python Variable

Assign the magic result to a variable with `<<` then convert to pandas with `.DataFrame()`.


In [11]:
# Save query result to a Python variable
result = %sql SELECT UniqueCarrier, COUNT(*) AS Flights, ROUND(AVG(ArrDelay),2) AS AvgDelay               FROM flights WHERE ArrDelay IS NOT NULL               GROUP BY UniqueCarrier HAVING Flights > 5000 ORDER BY AvgDelay;

# Convert to pandas DataFrame
import pandas as pd
df = result.DataFrame()
print(type(df))
df.head(10)

 * sqlite:///flights.db
Done.
<class 'pandas.core.frame.DataFrame'>


,UniqueCarrier,Flights,AvgDelay
0,US,38513,1.82
1,DL,36971,4.44
2,B6,16207,4.86
3,WN,100107,6.87
4,F9,7674,7.12
5,FL,20234,7.22
6,OH,18111,7.55
7,AS,12289,8.18
8,CO,24946,9.00
9,EV,21945,9.17


## Step 6 — Plot Directly from the Result

Once you have a DataFrame, use matplotlib or pandas `.plot()` as normal.


In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df_sorted = df.sort_values('AvgDelay')
colors = ['#2ca02c' if x <= 0 else '#d62728' for x in df_sorted['AvgDelay']]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(df_sorted['UniqueCarrier'], df_sorted['AvgDelay'], color=colors, alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Average Arrival Delay by Carrier (SQL Magic → Pandas → Matplotlib)')
ax.set_xlabel('Minutes')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_16008\3393806373.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 7 — JOINs with SQL Magic

`%%sql` handles multi-table JOINs just like any SQL client.


In [13]:
%%sql

-- Join flights → airports → carriers for a rich summary
SELECT
    f.Origin                     AS IATA,
    a.airport                    AS AirportName,
    a.city                       AS City,
    a.state                      AS State,
    c.Description                AS TopCarrier,
    COUNT(*)                     AS Departures,
    ROUND(AVG(f.DepDelay), 2)   AS AvgDepDelay
FROM flights f
LEFT JOIN airports a ON f.Origin   = a.iata
LEFT JOIN carriers c ON f.UniqueCarrier = c.Code
WHERE f.DepDelay IS NOT NULL
GROUP BY f.Origin
ORDER BY Departures DESC
LIMIT 10;

 * sqlite:///flights.db
Done.


IATA,AirportName,City,State,TopCarrier,Departures,AvgDepDelay
ATL,William B Hartsfield-Atlanta Intl,Atlanta,GA,Expressjet Airlines Inc.,32694,10.02
ORD,Chicago O'Hare International,Chicago,IL,Expressjet Airlines Inc.,27605,27.52
DFW,Dallas-Fort Worth International,Dallas-Fort Worth,TX,Expressjet Airlines Inc.,23233,11.39
DEN,Denver Intl,Denver,CO,Southwest Airlines Co.,19121,11.89
LAX,Los Angeles International,Los Angeles,CA,Southwest Airlines Co.,18540,13.97
PHX,Phoenix Sky Harbor International,Phoenix,AZ,Southwest Airlines Co.,17447,9.11
IAH,George Bush Intercontinental,Houston,TX,Expressjet Airlines Inc.,15407,9.61
LAS,McCarran International,Las Vegas,NV,Southwest Airlines Co.,15088,13.51
DTW,Detroit Metropolitan-Wayne County,Detroit,MI,Southwest Airlines Co.,14018,11.21
SLC,Salt Lake City Intl,Salt Lake City,UT,Southwest Airlines Co.,12169,12.36


## Step 8 — Subqueries & CTEs

CTEs (`WITH` clauses) work exactly the same in `%%sql`.


In [14]:
%%sql

-- CTE: identify high-delay carriers, then find their worst routes
WITH HighDelayCarriers AS (
    SELECT UniqueCarrier
    FROM flights
    WHERE ArrDelay IS NOT NULL
    GROUP BY UniqueCarrier
    HAVING AVG(ArrDelay) > 12
)
SELECT
    f.UniqueCarrier,
    f.Origin || ' → ' || f.Dest  AS Route,
    COUNT(*)                      AS Flights,
    ROUND(AVG(f.ArrDelay), 2)    AS AvgDelay
FROM flights f
INNER JOIN HighDelayCarriers h ON f.UniqueCarrier = h.UniqueCarrier
WHERE f.ArrDelay IS NOT NULL
GROUP BY f.UniqueCarrier, f.Origin, f.Dest
HAVING Flights > 100
ORDER BY AvgDelay DESC
LIMIT 12;

 * sqlite:///flights.db
Done.


UniqueCarrier,Route,Flights,AvgDelay
YV,ORD → CAE,105,60.83
OO,SLC → SFO,232,60.33
OO,RNO → SFO,132,59.03
OO,BOI → SFO,143,56.82
OO,ONT → SFO,123,55.59
OO,SBA → SFO,242,54.71
MQ,CMI → ORD,192,52.05
OO,ACV → SFO,190,51.91
OO,EUG → SFO,188,51.78
AA,LAS → ORD,145,50.51


## Step 9 — Cancellation Analysis with CASE Logic


In [15]:
%%sql

-- Cancellation rate and reason breakdown per carrier
SELECT
    UniqueCarrier                                        AS Carrier,
    COUNT(*)                                             AS TotalFlights,
    SUM(CAST(Cancelled AS INTEGER))                      AS Cancelled,
    ROUND(
        100.0 * SUM(CAST(Cancelled AS INTEGER)) / COUNT(*), 2
    )                                                    AS CancelRate,
    SUM(CASE WHEN CancellationCode = 'A' THEN 1 ELSE 0 END) AS CarrierFault,
    SUM(CASE WHEN CancellationCode = 'B' THEN 1 ELSE 0 END) AS WeatherFault,
    SUM(CASE WHEN CancellationCode = 'C' THEN 1 ELSE 0 END) AS NASFault
FROM flights
GROUP BY UniqueCarrier
HAVING TotalFlights > 5000
ORDER BY CancelRate DESC;

 * sqlite:///flights.db
Done.


Carrier,TotalFlights,Cancelled,CancelRate,CarrierFault,WeatherFault,NASFault
YV,22161,1440,6.5,787,368,285
MQ,43454,2447,5.63,459,998,990
OO,48992,2564,5.23,449,857,1258
9E,22848,1133,4.96,750,311,72
EV,23115,1128,4.88,134,989,5
UA,38026,1481,3.89,923,366,192
AA,52410,1657,3.16,814,342,501
DL,38256,1199,3.13,296,850,53
AS,12729,355,2.79,239,96,20
OH,18644,501,2.69,183,318,0


## Step 10 — Distance Bucketing with CASE


In [16]:
%%sql

-- Group flights into distance bands and compare delay profiles
SELECT
    CASE
        WHEN Distance < 500  THEN '① Short  (<500 mi)'
        WHEN Distance < 1000 THEN '② Medium (500–999 mi)'
        WHEN Distance < 2000 THEN '③ Long   (1000–1999 mi)'
        ELSE                      '④ X-Long (2000+ mi)'
    END                          AS DistanceBand,
    COUNT(*)                     AS Flights,
    ROUND(AVG(ArrDelay),  2)    AS AvgArrDelay,
    ROUND(AVG(DepDelay),  2)    AS AvgDepDelay,
    ROUND(AVG(ActualElapsedTime), 1) AS AvgFlightTime_min
FROM flights
WHERE ArrDelay IS NOT NULL
GROUP BY DistanceBand
ORDER BY DistanceBand;

 * sqlite:///flights.db
Done.


DistanceBand,Flights,AvgArrDelay,AvgDepDelay,AvgFlightTime_min
① Short (<500 mi),255479,10.63,10.96,74.9
② Medium (500–999 mi),199077,10.39,11.69,130.0
③ Long (1000–1999 mi),106753,9.39,12.01,204.2
④ X-Long (2000+ mi),25821,7.54,11.84,329.8



## Step 11 — Quick Stats with `%sql` One-liners


In [17]:
# Overall on-time rate
%sql SELECT ROUND(100.0 * SUM(CASE WHEN ArrDelay <= 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS OnTimePct FROM flights WHERE ArrDelay IS NOT NULL;

 * sqlite:///flights.db
Done.


OnTimePct
52.41


In [18]:
# Worst single flight delay
%sql SELECT UniqueCarrier, Origin, Dest, Date, ArrDelay FROM flights ORDER BY ArrDelay DESC LIMIT 5;

 * sqlite:///flights.db
Done.


UniqueCarrier,Origin,Dest,Date,ArrDelay
AA,EGE,MIA,2008/1/5,1525.0
AA,DEN,DFW,2008/1/16,1357.0
AA,SNA,ORD,2008/1/6,1147.0
NW,FLL,DTW,2008/1/28,1146.0
NW,SAT,MEM,2008/1/27,1123.0


In [19]:
# Average taxi-in vs taxi-out times
%sql SELECT ROUND(AVG(CAST(TaxiIn AS FLOAT)),2) AS AvgTaxiIn, ROUND(AVG(CAST(TaxiOut AS FLOAT)),2) AS AvgTaxiOut FROM flights;

 * sqlite:///flights.db
Done.


AvgTaxiIn,AvgTaxiOut
6.89,16.8


## flight analysis portfolio
ALx . Africa . Gideon kimani 